In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# Required
subscription_id = os.environ["SUBSCRIPTION_ID"]
tenant_id = os.environ["TENANT_ID"]
service_principal_id = os.environ["AZURE_SERVICE_PRINCIPAL_APPID"]
service_principal_password = os.environ["AZURE_SERVICE_PRINCIPAL_PASSWORD"]

# Optional
resource_group = os.getenv("RESOURCE_GROUP", default="demo-try-azureml")
workspace_name = os.getenv("WORKSPACE_NAME", default="demo-try-azureml")
workspace_region = os.getenv("WORKSPACE_REGION", default="eastus2")

print("Azure configuration loaded successfully!")

Azure configuration loaded successfully!


# Define the Service Principal

In [ ]:
%pip install azureml-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached azureml_core-1.61.0.post4-py3-none-any.whl.metadata (3.6 kB)
  Using cached pytz-2026.6-py2.py3-none-any.whl.metadata (22 kB)
  Using cached backports.tempfile-1.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached pathspec-0.12.1-py3-none-any.whl.metadata (21 kB)
  Using cached msal-1.39.0-py3-none-any.whl.metadata (11 kB)
  Using cached msal_extensions-1.3.1-py3-none-any.whl.metadata (7.8 kB)
  Using cached knack-0.12.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached azure_core-1.41.0-py3-none-any.whl.metadata (49 kB)
  Using cached pkginfo-2.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached argcomplete-3.7.2-py3-none-any.whl.metadata (16 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
  Using cached paramiko-3.5.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached azure_mgmt_resource-24.0.0-py3-none-any.whl.metadata (43 kB)
  Using cached azure_mgmt_containerregistry-14.0.0-py3-none-any.whl.metadata (25 kB)
  Using cached azure_mgmt_

In [5]:
from azureml.core import Workspace
from azureml.core.authentication import ServicePrincipalAuthentication

service_principal = ServicePrincipalAuthentication(
    tenant_id=tenant_id,
    service_principal_id=service_principal_id,
    service_principal_password=service_principal_password
)

# Creating the workspace

In [8]:
import socket

print(socket.gethostbyname("management.azure.com"))

4.150.240.10


In [9]:
# Create the workspace using the specified parameters
ws = Workspace.create(
    name=workspace_name,
    subscription_id=subscription_id,
    resource_group=resource_group,
    location=workspace_region,
    auth=service_principal,
    create_resource_group=True,
    exist_ok=True
)

ws.get_details()

# Write the details of the workspace to a configuration file
# in the notebook library

ws.write_config()


Deploying StorageAccount with name demotryastorage00da1dd3b.
Deploying KeyVault with name demotryakeyvault75e33943.
Deployed KeyVault with name demotryakeyvault75e33943. Took 94.43 seconds.
Deploying Workspace with name demo-try-azureml.
Deploying AppInsights with name demotryainsightsa5bf487c.
Deployed AppInsights with name demotryainsightsa5bf487c. Took 1.79 seconds.
Deployed Workspace with name demo-try-azureml. Took 18.22 seconds.


In [10]:
from azureml.core.compute import ComputeTarget, AmlCompute

# Define all compute target(s) to create
amlcomputes = {
    "cpu-cluster": {
        "vm_size": "STANDARD_DS3_V2",
        "min_nodes": 0,
        "max_nodes": 3,
        "idle_seconds_before_scaledown": 240,
    }
}

# Create all compute targets
for ct_name in amlcomputes:
    if ct_name not in ws.compute_targets:
        compute_config = AmlCompute.provisioning_configuration(
            **amlcomputes[ct_name]
        )
        ct = ComputeTarget.create(ws, ct_name, compute_config)
        ct.wait_for_completion(show_output=True)

InProgress....
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


# Exploratory work

In [11]:
# Exploratory work

# 1. Display workspace details
print("Workspace Name:", ws.name)
print("Resource Group:", ws.resource_group)
print("Workspace Location:", ws.location)
print("Subscription ID:", ws.subscription_id)

# 2. List available compute targets
print("\nAvailable Compute Targets:")

for compute_name, compute_target in ws.compute_targets.items():
    print(compute_name, ":", compute_target.type)

# 3. Inspect the CPU cluster
cpu_cluster = ws.compute_targets["cpu-cluster"]

print("\nCPU Cluster Details:")
print("Name:", cpu_cluster.name)
print("Type:", cpu_cluster.type)
print("Provisioning State:", cpu_cluster.provisioning_state)

# 4. Display cluster properties
print("\nCluster Configuration:")
print("VM Size:", cpu_cluster.serialize()["properties"]["properties"]["vmSize"])


Workspace Name: demo-try-azureml
Resource Group: demo-try-azureml
Workspace Location: eastus2
Subscription ID: 8a084f2d-2327-478f-8b59-1fd30dcbdfd3

Available Compute Targets:
cpu-cluster : AmlCompute

CPU Cluster Details:
Name: cpu-cluster
Type: AmlCompute
Provisioning State: Succeeded

Cluster Configuration:
VM Size: STANDARD_DS3_V2


In [12]:
ws.delete(delete_dependent_resources=True, no_wait=False)